# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [2]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): github.com/yoonji57/gpt-lab.git
GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ··········
Repo: /content/gpt-lab


In [3]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [4]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
저장됨: /content/gpt-lab/data/ratings_train.txt
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
저장됨: /content/gpt-lab/data/ratings_test.txt
사전 학습 train 텍스트: /content/gpt-lab/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /content/gpt-lab/data/nsmc_lm_val.txt (120,560자)
감성 분류 train: /content/gpt-lab/data/nsmc_sentiment_train.jsonl (137,996개)
감성 분류 val: /content/gpt-lab/data/nsmc_sentiment_val.jsonl (11,999개)
감성 분류 test: /content/gpt-lab/data/nsmc_sentiment_test.jsonl (49,997개)
LM train exists: True /content/gpt-lab/data/nsmc_lm_train.txt
LM val exists: True /content/gpt-lab/data/nsmc_lm_val.txt


In [5]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

train chars: 1379486
val chars: 120560
개재미없다. 감독의 연출력의 한계
이제서야 보게된 대 명작 연출미가 정말 훌륭하다!!!!!!!!
소주미라클을 만들어라
귀여운 캐릭터들도 많이 나와서 보러 가야 겠어요..
블랙 코미디가 싫어요.
평점깎고싶다10글자
TV시리즈가 너무재밌어서 영화는 기대안하고 봤는데 역시....최고네요
개인적 공감이 글쎄?
시작은 니시지마 때문에 봤는데 나름 괜찮은 영화 봤다고


## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [6]:
run_pytest("tests/test_bpe.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_bpe.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/gpt-lab
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collecting ... collected 6 items

tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 16%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 33%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 50%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 66%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 83%]
tests/test_bpe.py::TestBPETrain::test_train_increases_vocab PASSED       [100%]

============================== 6 passed in 0.05s ===============================


선택한 테스트를 통과했습니다.


0

In [7]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

[2, 240, 161, 184, 36, 240, 156, 133, 241, 157, 152, 239, 142, 152, 36, 240, 164, 153, 239, 171]
이 영화는 정말 좋았다! English 123


## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [8]:
run_pytest("tests/test_dataset.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_dataset.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/gpt-lab
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collecting ... collected 4 items

tests/test_dataset.py::TestGPTDataset::test_dataset_length PASSED        [ 25%]
tests/test_dataset.py::TestGPTDataset::test_dataset_getitem_shape PASSED [ 50%]
tests/test_dataset.py::TestCreateDataloader::test_dataloader_batch_shape PASSED [ 75%]
tests/test_dataset.py::TestInputEmbedding::test_input_embedding_shape PASSED [100%]

============================== 4 passed in 3.15s ===============================


선택한 테스트를 통과했습니다.


0

In [9]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

torch.Size([2, 32]) torch.Size([2, 32]) torch.Size([2, 32, 32])


## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [10]:
run_pytest("tests/test_attention.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_attention.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/gpt-lab
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collecting ... collected 2 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [ 50%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [100%]

============================== 2 passed in 2.04s ===============================


선택한 테스트를 통과했습니다.


0

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [11]:
run_pytest("tests/test_model.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_model.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/gpt-lab
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collecting ... collected 7 items

tests/test_model.py::TestLayerNorm::test_layernorm_shape PASSED          [ 14%]
tests/test_model.py::TestGELU::test_gelu_shape PASSED                    [ 28%]
tests/test_model.py::TestFeedForward::test_feedforward_shape PASSED      [ 42%]
tests/test_model.py::TestTransformerBlock::test_transformer_block_shape PASSED [ 57%]
tests/test_model.py::TestGPTModel::test_gpt_forward_shape PASSED         [ 71%]
tests/test_model.py::TestGPTModel::test_gpt_forward_with_targets_returns_loss PASSED [ 85%]
tests/test_model.py::TestGenerateTextSimple::test_generate_text_simple_shape PASSED [100%]

============================== 7 passed in 2.06s ==

0

In [12]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

torch.Size([2, 16, 300])


## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [13]:
run_pytest("tests/test_train.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_train.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/gpt-lab
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collecting ... collected 5 items

tests/test_train.py::TestCalcLossBatch::test_calc_loss_batch_returns_scalar PASSED [ 20%]
tests/test_train.py::TestCalcLossLoader::test_calc_loss_loader_returns_float PASSED [ 40%]
tests/test_train.py::TestCheckpoint::test_save_load_checkpoint_restores_epoch_and_step PASSED [ 60%]
tests/test_train.py::TestGenerate::test_generate_shape PASSED            [ 80%]
tests/test_train.py::TestPlotLosses::test_plot_losses_callable PASSED    [100%]

============================== 5 passed in 12.01s ==============================


선택한 테스트를 통과했습니다.


0

In [14]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

smoke loss: 5.879581928253174


In [17]:
# 기존 vocab JSON을 사용해서 실제 학습을 돌리는 셀입니다.
# 처음에는 MAX_*_CHARS와 max_steps를 작게 두고 동작을 확인하세요.
from datetime import datetime, timedelta, timezone
import platform
import time

import matplotlib.pyplot as plt
import torch

from bpe import BPETokenizer
from dataset import create_dataloader
from model import GPTModel
from train import calc_loss_batch, calc_loss_loader, generate

KST = timezone(timedelta(hours=9))
run_started_at = datetime.now(KST)
run_id = run_started_at.strftime(“%Y%m%d_%H%M%S”)
run_timer_start = time.perf_counter()

device = torch.device(“cuda” if torch.cuda.is_available() else “cpu”)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else “CPU”
print(“device:“, device)
print(“environment:“, gpu_name)
print(“started at:“, run_started_at.strftime(“%Y-%m-%d %H:%M:%S KST”))

# 1. 기존 vocab 로드: 여기서 tokenizer.train(...)은 호출하지 않습니다.
VOCAB_PATH = repo_dir / “data” / “bpe_tokenizer_vocab3000.json”
if not VOCAB_PATH.exists():
    raise FileNotFoundError(f”vocab file not found: {VOCAB_PATH}“)

tokenizer = BPETokenizer()
tokenizer.load(VOCAB_PATH)
vocab_size = len(tokenizer.id_to_token)
print(“loaded vocab size:“, vocab_size)

# 2. corpus를 token id로 변환합니다.
# 전체 corpus를 쓰기 전에 작은 범위로 먼저 확인하세요.
MAX_TRAIN_CHARS = 200_0000
MAX_VAL_CHARS = 20_0000

train_ids = tokenizer.encode(corpus[:MAX_TRAIN_CHARS])
val_ids = tokenizer.encode(val_corpus[:MAX_VAL_CHARS])
print(“train tokens:“, len(train_ids))
print(“val tokens:“, len(val_ids))

# 3. DataLoader 생성
context_length = 64
batch_size = 16
stride = context_length

train_loader = create_dataloader(
    train_ids,
    context_length=context_length,
    batch_size=batch_size,
    stride=stride,
    drop_last=True,
    shuffle=True,
)
val_loader = create_dataloader(
    val_ids,
    context_length=context_length,
    batch_size=batch_size,
    stride=stride,
    drop_last=True,
    shuffle=False,
)
print(“steps per epoch:“, len(train_loader))

# 4. 모델 생성: vocab_size는 3000이 아니라 로드된 실제 vocab 크기를 사용합니다.
config = {
    “vocab_size”: vocab_size,
    “context_length”: context_length,
    “emb_dim”: 128,
    “n_heads”: 4,
    “n_layers”: 2,
    “drop_rate”: 0.1,
    “qkv_bias”: False,
}

model = GPTModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
learning_rate = optimizer.param_groups[0][“lr”]

# 5. 실제 학습 루프
num_epochs = 5
eval_freq = 100
eval_iter = 3
max_steps = 2000
global_step = 0

eval_steps = []
train_losses = []
val_losses = []


def record_losses(step):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    eval_steps.append(step)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f”step {step}: train loss {train_loss:.3f}, val loss {val_loss:.3f}“)


for epoch in range(num_epochs):
    model.train()
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        loss = calc_loss_batch(input_batch, target_batch, model, device)
        loss.backward()
        optimizer.step()

        if global_step % eval_freq == 0:
            record_losses(global_step)

        global_step += 1
        if global_step >= max_steps:
            break
    if global_step >= max_steps:
        break

# 마지막 step 손실도 한 번 더 기록해서 짧은 smoke run에서도 그래프가 보이게 합니다.
if not eval_steps or eval_steps[-1] != global_step:
    record_losses(global_step)

elapsed_sec = time.perf_counter() - run_timer_start
finished_at = datetime.now(KST)
print(“training done. global_step:“, global_step)
print(“finished at:“, finished_at.strftime(“%Y-%m-%d %H:%M:%S KST”))
print(f”elapsed: {elapsed_sec / 60:.1f} min”)

# 6. 조건이 적힌 loss 그래프를 PNG로 저장하고 Colab이면 다운로드합니다.
result_dir = repo_dir / “output” / “training_runs”
result_dir.mkdir(parents=True, exist_ok=True)
plot_path = result_dir / f”loss_{run_id}_steps{global_step}.png”

final_train_loss = train_losses[-1] if train_losses else float(“nan”)
final_val_loss = val_losses[-1] if val_losses else float(“nan”)

run_conditions = [
    f”started: {run_started_at.strftime(‘%Y-%m-%d %H:%M:%S KST’)}“,
    f”finished: {finished_at.strftime(‘%Y-%m-%d %H:%M:%S KST’)}  elapsed: {elapsed_sec / 60:.1f} min”,
    f”device: {device}  gpu: {gpu_name}“,
    f”python: {platform.python_version()}  torch: {torch.__version__}“,
    f”vocab: {VOCAB_PATH.name}  vocab_size: {vocab_size:,}“,
    f”chars: train={MAX_TRAIN_CHARS:,}  val={MAX_VAL_CHARS:,}“,
    f”tokens: train={len(train_ids):,}  val={len(val_ids):,}“,
    f”loader: context={context_length}  stride={stride}  batch={batch_size}  steps/epoch={len(train_loader)}“,
    f”train: epochs={num_epochs}  max_steps={max_steps}  actual_steps={global_step}  eval_freq={eval_freq}  eval_iter={eval_iter}“,
    f”optimizer: AdamW  lr={learning_rate:g}“,
    f”model: emb={config[‘emb_dim’]}  heads={config[‘n_heads’]}  layers={config[‘n_layers’]}  dropout={config[‘drop_rate’]}“,
    f”final loss: train={final_train_loss:.3f}  val={final_val_loss:.3f}“,
]

fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(eval_steps, train_losses, marker=“o”, label=“train”)
ax.plot(eval_steps, val_losses, marker=“o”, label=“val”)
ax.set_xlabel(“global step”)
ax.set_ylabel(“loss”)
ax.set_title(“Training / Validation Loss”)
ax.grid(alpha=0.3)
ax.legend()
fig.subplots_adjust(bottom=0.38)
fig.text(
    0.02,
    0.02,
    “\n”.join(run_conditions),
    ha=“left”,
    va=“bottom”,
    fontsize=8,
    family=“monospace”,
    bbox={“boxstyle”: “round,pad=0.45”, “facecolor”: “#F8FAFC”, “edgecolor”: “#CBD5E1”},
)
fig.savefig(plot_path, dpi=180, bbox_inches=“tight”)
plt.show()
print(“saved loss plot:“, plot_path)

try:
    from google.colab import files

    files.download(str(plot_path))
except Exception as e:
    print(“Colab download skipped:“, e)

# 7. checkpoint 저장
CHECKPOINT_PATH = repo_dir / “checkpoints” / “gpt_vocab_loaded.pt”
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        “model_state_dict”: model.state_dict(),
        “config”: config,
        “vocab_path”: str(VOCAB_PATH),
        “global_step”: global_step,
        “eval_steps”: eval_steps,
        “train_losses”: train_losses,
        “val_losses”: val_losses,
        “plot_path”: str(plot_path),
        “run_conditions”: run_conditions,
    },
    CHECKPOINT_PATH,
)
print(“saved:“, CHECKPOINT_PATH)

# 8. 간단한 샘플 생성
model.eval()
start_text = “이 영화는”
start_ids = tokenizer.encode(start_text)
idx = torch.tensor(start_ids, dtype=torch.long).unsqueeze(0).to(device)
with torch.no_grad():
    out = generate(
        model=model,
        idx=idx,
        max_new_tokens=50,
        context_size=context_length,
        temperature=0.8,
        top_k=40,
        eos_id=tokenizer.get_eos_id(),
    )
try:
    print(tokenizer.decode(out[0].tolist()))
except UnicodeDecodeError as e:
    print(“generated ids could not be decoded yet:“, e)
    print(out[0].tolist())

SyntaxError: invalid character '“' (U+201C) (3224026267.py, line 17)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [15]:
run_pytest("tests/test_finetune.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_finetune.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/gpt-lab
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collecting ... collected 4 items

tests/test_finetune.py::TestMakeSentimentDataset::test_make_sentiment_dataset_splits_rows FAILED [ 25%]
tests/test_finetune.py::TestReviewSentimentDataset::test_review_sentiment_dataset_getitem FAILED [ 50%]
tests/test_finetune.py::TestGPTForSequenceClassification::test_sequence_classification_shape FAILED [ 75%]
tests/test_finetune.py::TestSentimentTrainEval::test_train_eval_functions_exist PASSED [100%]

=================================== FAILURES ===================================
_______ TestMakeSentimentDataset.test_make_sentiment_dataset_splits_rows _______

self = <test_finetune.TestMakeSentimentDataset object at 0x7b1d31

1

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [16]:
run_pytest("tests/")

실행 명령: /usr/bin/python3 -m pytest tests/ -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/gpt-lab
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.5
collecting ... collected 28 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [  3%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [  7%]
tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 10%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 14%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 17%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 21%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 25%]
tests/test_bpe.py::TestBPETrain::test_train_increases_voca

1